<a href="https://colab.research.google.com/github/sparsetrace/DMRG/blob/main/DMRG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# basic imports

In [1]:
import os, sys, subprocess as sbp
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
try:
    import dmrg
except:
    sbp.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/sparsetrace/DMRG.git"])
    import dmrg

# DMRG surgery

In [5]:
from dmrg import DMRG, DiffusionBlock, VIT_metrics

metrics_viu = VIT_metrics("jcandane/ViU", subfolder="viu", max_images=512, trust_remote_code=True)

print(metrics_viu)

A new version of the following files was downloaded from https://huggingface.co/jcandane/ViU:
- modeling_viu.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

{'acc1': 0.8203125, 'acc5': 0.95703125, 'xent': 0.7459716796875, 'n': 512, 'params_total': 83023337}


In [3]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification

from dmrg import DMRG, DiffusionBlock, VIT_metrics

#HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_HUB_TOKEN")
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

# 1) Load processor + base model from the HF repo/subfolder
processor = AutoImageProcessor.from_pretrained(
    "jcandane/ViU",
    subfolder="vit",
    trust_remote_code=False,
    token=HF_TOKEN,
)

model = AutoModelForImageClassification.from_pretrained(
    "jcandane/ViU",
    subfolder="vit",
    trust_remote_code=False,
    token=HF_TOKEN,
)

# 2) Build a train loader
ds = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True, token=True)

def collate_fn(examples):
    images = []
    labels = []
    for ex in examples:
        img = ex["image"]
        if hasattr(img, "convert"):
            img = img.convert("RGB")
        images.append(img)
        labels.append(int(ex["label"]))

    batch = processor(images=images, return_tensors="pt")
    batch["labels"] = __import__("torch").tensor(labels, dtype=__import__("torch").long)
    return batch

train_loader = DataLoader(
    ds,
    batch_size=32,
    collate_fn=collate_fn,
)

# 3) Initialize DMRG
dmrg = DMRG(
    model=model,
    train_loader=train_loader,
    teacher="self",      # frozen copy of the original model
    hf_token=HF_TOKEN,
)

# 4a) Smoke test: first window only
dmrg.run_first_window(
    new_block=DiffusionBlock,
    block_kwargs={
        "channel_heads": 8,
        "dropout": 0.0,
    },
    steps_per_window=100,
    output_dir="./dmrg_ckpts",
    repo_id="jcandane/ViU",
    hub_path="dmrg_test_first_window",
)

# 4b) Full run
# dmrg.run(
#     new_block=DiffusionBlock,
#     block_kwargs={
#         "channel_heads": 8,
#         "dropout": 0.0,
#     },
#     sweeps=1,
#     steps_per_window=500,
#     output_dir="./dmrg_ckpts",
#     repo_id="jcandane/ViU",
#     hub_path="dmrg_full_run",
# )

config.json: 0.00B [00:00, ?B/s]

vit/model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_10_11/model.safetensors:  23%|##3       | 73.7MB /  318MB            

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-9): 10 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermedi